In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from dataset import FaceDataset


In [45]:
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR="/kaggle/input/fdsdfdsdfg/faces_split/faces_split"
BATCH_SIZE=32
EPOCHS=5
LR=1e-3

In [ ]:
transform=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor()])
train_ds=FaceDataset(DATA_DIR + "/train",transform)
val_ds=FaceDataset(DATA_DIR + "/val",transform)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(nn.Conv2d(3,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),)
        self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(32*56*56,1))
    def forward(self,x):
        x=self.features(x)
        return self.classifier(x)

In [ ]:
model=SimpleCNN().to(DEVICE)
criterion=nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters(),lr=LR)
for epoch in range(EPOCHS):
    model.train()
    total_loss=0
    for imgs,labels in train_loader:
        imgs,labels=imgs.to(DEVICE),labels.float().to(DEVICE)
        optimizer.zero_grad()
        outputs=model(imgs).squeeze()
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
torch.save(model.state_dict(),"baseline_cnn.pth")
print("✅ Baseline CNN trained & saved")
